In [1]:
import numpy as np


# ACTIVATION FUNCTIONS

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def softmax(x):
    exp_x = np.exp(x - np.max(x))
    return exp_x / np.sum(exp_x)


# CONVOLUTION LAYER

class ConvLayer:

    def __init__(self, num_filters, filter_size):

        self.num_filters = num_filters
        self.filter_size = filter_size

        self.filters = (
            np.random.randn(
                num_filters,
                filter_size,
                filter_size
            ) / (filter_size * filter_size)
        )

    def forward(self, image):

        self.image = image

        h, w = image.shape

        output = np.zeros(
            (
                h - self.filter_size + 1,
                w - self.filter_size + 1,
                self.num_filters
            )
        )

        for f in range(self.num_filters):

            current_filter = self.filters[f]

            for i in range(h - self.filter_size + 1):

                for j in range(w - self.filter_size + 1):

                    region = image[
                        i:i+self.filter_size,
                        j:j+self.filter_size
                    ]

                    output[i, j, f] = np.sum(
                        region * current_filter
                    )

        return output



# MAX POOLING

class MaxPool:

    def __init__(self, pool_size=2):

        self.pool_size = pool_size

    def forward(self, input_data):

        self.input_data = input_data

        h, w, num_filters = input_data.shape

        output = np.zeros(
            (
                h // self.pool_size,
                w // self.pool_size,
                num_filters
            )
        )

        for f in range(num_filters):

            for i in range(0, h, self.pool_size):

                for j in range(0, w, self.pool_size):

                    region = input_data[
                        i:i+self.pool_size,
                        j:j+self.pool_size,
                        f
                    ]

                    output[
                        i//self.pool_size,
                        j//self.pool_size,
                        f
                    ] = np.max(region)

        return output



# FULLY CONNECTED LAYER

class Dense:

    def __init__(self, input_size, output_size):

        self.weights = (
            np.random.randn(
                input_size,
                output_size
            ) * 0.01
        )

        self.bias = np.zeros((1, output_size))

    def forward(self, x):

        self.input = x

        return np.dot(x, self.weights) + self.bias



# CNN MODEL

class CNN:

    def __init__(self):

        self.conv = ConvLayer(
            num_filters=8,
            filter_size=3
        )

        self.pool = MaxPool(
            pool_size=2
        )

        self.fc = Dense(
            input_size=13 * 13 * 8,
            output_size=10
        )

    def forward(self, image):

        conv_output = self.conv.forward(image)

        relu_output = relu(conv_output)

        pooled_output = self.pool.forward(
            relu_output
        )

        flattened = pooled_output.flatten()

        logits = self.fc.forward(
            flattened.reshape(1, -1)
        )

        probabilities = softmax(logits)

        return probabilities


In [2]:

# EXAMPLE

if __name__ == "__main__":

    # Example image (28x28 grayscale)

    image = np.random.rand(28, 28)

    cnn = CNN()

    output = cnn.forward(image)

    print("Class Probabilities:")
    print(output)

    print("\nPredicted Class:")
    print(np.argmax(output))

Class Probabilities:
[[0.10073527 0.1048517  0.09753117 0.1078674  0.0889178  0.08764896
  0.09658323 0.1147551  0.09877505 0.10233433]]

Predicted Class:
7
